In [ ]:
# =========================
# 3. Preprocessing
# =========================

def clean_number(value):
    if pd.isna(value):
        return np.nan

    value = str(value)
    value = value.replace('\n', ' ')
    value = value.replace('\r', ' ')
    value = value.replace(',', '')
    value = value.replace('EGP', '')
    value = value.replace('جنيه', '')
    value = value.replace('..', '.')
    value = value.strip()

    return pd.to_numeric(value, errors='coerce')


numeric_cols = [
    'seller_rating',
    'product_rating',
    'seller_reviews_count',
    'discount_percent',
    'price',
    'original_price'
]

for col in numeric_cols:
    if col in data.columns:
        data[col] = data[col].apply(clean_number)


text_cols = ['title', 'description', 'reviews', 'seller_name', 'image_url']

for col in text_cols:
    if col in data.columns:
        data[col] = (
            data[col]
            .astype(str)
            .str.replace('\n', ' ', regex=False)
            .str.replace('\r', ' ', regex=False)
            .str.replace(',', ' ', regex=False)
            .str.strip()
        )


data['discount_percent'] = data['discount_percent'].fillna(0)
data['price'] = data['price'].fillna(0)
data['original_price'] = data['original_price'].fillna(0)

data['price_diff'] = np.where(
    (data['price'] > 0) & (data['original_price'] > 0),
    data['original_price'] - data['price'],
    0
)

data['real_discount'] = np.where(
    (data['price'] > 0) & (data['original_price'] > 0),
    (data['price_diff'] / data['original_price']) * 100,
    0
)

data['real_discount'] = (
    data['real_discount']
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

data = data.dropna(subset=[
    'seller_rating',
    'seller_reviews_count',
    'product_rating'
]).reset_index(drop=True)

print(data[['price', 'original_price', 'price_diff', 'real_discount']].head(10))
print("Data shape after cleaning:", data.shape)

Seller Model

In [ ]:
seller_features = [
    'seller_rating',
    'seller_reviews_count',
    'product_rating',
    'real_discount'
]

X_seller = data[seller_features]
y_seller = data['label'].astype(int)

X_seller_train, X_seller_test, y_seller_train, y_seller_test = train_test_split(
    X_seller,
    y_seller,
    test_size=0.2,
    random_state=42,
    stratify=y_seller
)

model_seller = RandomForestClassifier(n_estimators=50, random_state=42)
model_seller.fit(X_seller_train, y_seller_train)

seller_pred = model_seller.predict(X_seller_test)
print("Seller Model Accuracy:", accuracy_score(y_seller_test, seller_pred))
